# Who Gets Remembered? Validation check and oral-defence prep

Reads directly from the repo files (`data_collection/` and `pipeline_final/`). Put this notebook in `pipeline_final/notebooks/` (or anywhere inside the repo) and run top to bottom.

1. **Project at a glance**: headline numbers, read from `CONCLUSIONS.md` and the results files
2. **The match check**: what the existing 100-match check really is, and how to do the *human* version
3. **Scoring**: accuracy with Wilson intervals by gender, agreement between two labellers
4. **The 300-sentence check**
5. **Oral-defence self-test**

Requirements: `pandas numpy scipy scikit-learn` (already in `pipeline_final/requirements.txt`).

In [1]:
import os, re, json, random, webbrowser, subprocess, sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact
from sklearn.metrics import cohen_kappa_score
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", 90, "display.width", 160)

def find_repo(start=None):
    """Walk up from the notebook folder until a folder with both data_collection/ and pipeline_final/ is found."""
    if os.environ.get("REPO_PATH"):
        return Path(os.environ["REPO_PATH"]).resolve()
    p = Path(start or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "data_collection").is_dir() and (q / "pipeline_final").is_dir():
            return q
    raise FileNotFoundError("Repo root not found. Open VS Code in the repo folder, or set REPO_PATH.")

REPO = find_repo()
OLD, NEW = REPO / "data_collection", REPO / "pipeline_final"
PROC, VAL, RES = OLD / "data" / "processed", NEW / "validation", NEW / "results"
print("Repo:", REPO)

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    p = k / n
    c = (p + z*z/(2*n)) / (1 + z*z/n)
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / (1 + z*z/n)
    return (c - h, c + h)

Repo: C:\Users\35191\Desktop\Técnico\5º Ano\Erasmus\Social Graphs and Interactions\Projeto\SocialGraphs-Interactions


## 1. The project at a glance

In [2]:
txt = (NEW / "CONCLUSIONS.md").read_text(encoding="utf-8")
one = re.search(r"## One sentence\s+(.*?)\n## ", txt, re.S).group(1)
display(Markdown("**One-sentence result (from CONCLUSIONS.md):**\n\n" + one))

nodes = pd.read_csv(PROC / "nodes.csv")
comp = pd.read_csv(RES / "rq1_comparison_set.csv")
print(f"All pages: {len(nodes):,} | comparison set: {len(comp):,} "
      f"({(comp.gender=='female').sum()} women, {(comp.gender=='male').sum()} men)")
display(pd.read_csv(RES / "rq1_robustness.csv").round(3).head(12))

**One-sentence result (from CONCLUSIONS.md):**

At equal citations, field and birth decade, women scientists' Wikipedia pages receive about 40% fewer links from other scientists' pages than men's (rate ratio 0.60, 95% CI [0.51, 0.70]), give fewer links themselves, and are more often cut off from the network altogether; their text is also recognisably different even with names and gender words hidden.

The claim is **less embedded**, not *less visible*: we measure links between scientists' pages, not readers.


All pages: 12,463 | comparison set: 2,926 (606 women, 2320 men)


,variant,n,women,in_degree_ratio_perm,z,in_degree_irr_nb,irr_lo,irr_hi,out_degree_ratio_perm,isolate_ratio_perm
0,Main: all wikilinks,2926,606,0.661,-4.678,0.597,0.505,0.704,0.749,1.195
1,Prose-only links,2926,606,0.555,-4.477,0.475,0.384,0.588,0.778,1.154
2,Subfield strata + subfield control,2926,606,0.723,-3.668,0.646,0.549,0.761,0.810,1.096
3,Impact = h-index (largest profile),2926,606,0.652,-4.938,0.574,0.485,0.680,0.731,1.213
4,Single OpenAlex profile only,2491,531,0.661,-4.532,0.584,0.488,0.699,0.718,1.220
5,Without Nobel laureates,2830,600,0.684,-4.573,0.606,0.514,0.715,0.752,1.200
6,Without top 1% by citations,2896,603,0.669,-4.775,0.601,0.508,0.710,0.742,1.198
7,Born 1950 or later,2090,538,0.674,-4.066,0.608,0.506,0.731,0.768,1.169
8,Living people only,2702,579,0.661,-4.728,0.606,0.512,0.717,0.753,1.190


## 2. The match check: what exists, and what is missing

**What the question is.** Every scientist's citations come from an OpenAlex author profile found via ORCID/OpenAlex ID on Wikidata. The check asks: *is that profile the same person as the Wikipedia page?* If not, "equal citations" is wrong for that person. It also matters by gender: if matching were worse for women, the main result would be biased.

**What exists in the repo.** `data_collection/data/processed/handcheck_sample.csv`: 100 matches (50 women, 50 men) labelled **with AI help**. Your own `r5_validation.py` calls this circular (a model checking a model), so the paper must say so, and the plan is two *blind* human labellers.

In [3]:
ai = pd.read_csv(PROC / "handcheck_sample.csv")
rows = []
for g, d in ai.groupby("gender"):
    k, n = int(d.correct.sum()), len(d)
    lo, hi = wilson(k, n)
    rows.append({"gender": g, "n": n, "correct": k, "accuracy": k/n, "wilson_lo": lo, "wilson_hi": hi,
                 "profile_complete": d.profile_complete.mean()})
k, n = int(ai.correct.sum()), len(ai); lo, hi = wilson(k, n)
rows.append({"gender": "all", "n": n, "correct": k, "accuracy": k/n, "wilson_lo": lo, "wilson_hi": hi,
             "profile_complete": ai.profile_complete.mean()})
display(pd.DataFrame(rows).round(3))
print("Match methods in the sample:", ai.match_method.value_counts().to_dict())

,gender,n,correct,accuracy,wilson_lo,wilson_hi,profile_complete
0,female,50,50,1.0,0.929,1.0,0.98
1,male,50,50,1.0,0.929,1.0,0.94
2,all,100,100,1.0,0.963,1.0,0.96


Match methods in the sample: {'orcid': 98, 'openalex_id': 2}


Read this table honestly: **100/100 has a 95% Wilson interval of about 96–100%**, and per gender (50/50) about 93–100%. With 50 per group you cannot detect a difference in accuracy smaller than roughly 10 points, so the paper should report the intervals, not "no bias".

### How to do the human check (about one hour per person)

Two people each take a sheet (`matches_labeller_A.csv` / `_B.csv` in `pipeline_final/validation/`). The sheets hide gender and the AI labels on purpose, and have a different row order per person. **Do not look at each other's sheet or at the AI labels first.**

For each row open the Wikipedia link and the OpenAlex link, then enter:
- `correct`: 1 if the OpenAlex profile is the same person, else 0
- `profile_complete`: 1 if the profile looks like the whole career (not split or partial), else 0

Check name, field, institution, and that publication years fit the birth year. Be suspicious of very large or very small paper counts.

**Option 1:** run the cell below (it opens both links in your browser and asks for 1/0 in the notebook; progress is saved after every row, so you can stop and resume).
**Option 2:** just edit the CSV in VS Code or Excel.

In [4]:
WHO = "A"        # <- each labeller sets their own letter: "A" or "B"

def label(who, kind="matches", open_links=True):
    cols = {"matches": ["correct", "profile_complete"], "sentences": ["is_personal", "is_professional"]}[kind]
    f = VAL / f"{kind}_labeller_{who}.csv"
    if not f.exists():
        subprocess.run([sys.executable, str(NEW / "src" / "r5_validation.py"), "make"], cwd=NEW / "src", check=True)
    df = pd.read_csv(f, dtype=str)
    for c in cols + (["comment"] if kind == "matches" else []):
        if c not in df: df[c] = np.nan
    todo = df.index[df[cols].isna().any(axis=1)]
    print(f"{f.name}: {len(df) - len(todo)}/{len(df)} done, {len(todo)} left.  (1 = yes, 0 = no, s = skip, q = quit)")
    try:
        for i in todo:
            r = df.loc[i]
            print("\n" + "-"*70, f"\nrow {i+1}/{len(df)}")
            if kind == "matches":
                print(f"{r.title}  (b. {r.birth_year}, {r.field})   OpenAlex name: {r.oa_name}\n{r.wikipedia_url}\n{r.openalex_url}")
                if open_links:
                    webbrowser.open(r.wikipedia_url); webbrowser.open(r.openalex_url)
                prompts = {"correct": "Same person?", "profile_complete": "Profile looks like the whole career?"}
            else:
                print(r.sentence)
                prompts = {"is_personal": "Personal (family, marriage, children, private life)?",
                           "is_professional": "Professional (research, jobs, degrees, prizes)?"}
            for c in cols:
                if pd.notna(df.at[i, c]): continue
                while True:
                    a = input(f"  {prompts[c]} [1/0/s/q]: ").strip().lower()
                    if a in ("1", "0"): df.at[i, c] = a; break
                    if a == "s": break
                    if a == "q": df.to_csv(f, index=False); print("saved, bye"); return
            if kind == "matches":
                cm = input("  comment (optional, Enter to skip): ").strip()
                if cm: df.at[i, "comment"] = cm
            df.to_csv(f, index=False)
    except Exception as e:                       # e.g. no stdin when the notebook is run non-interactively
        df.to_csv(f, index=False)
        print("Stopped:", type(e).__name__, "- progress saved. Edit the CSV directly or re-run in VS Code.")
        return
    print("\nAll rows done for", f.name)

label(WHO, "matches")

matches_labeller_A.csv: 0/100 done, 100 left.  (1 = yes, 0 = no, s = skip, q = quit)

---------------------------------------------------------------------- 
row 1/100
Ewa Bulska  (b. 1953, chemistry)   OpenAlex name: Ewa Bulska
https://en.wikipedia.org/wiki/Ewa_Bulska
https://openalex.org/A5067303076

---------------------------------------------------------------------- 
row 2/100
Réka Albert  (b. 1972, biology|physics)   OpenAlex name: Réka Albert
https://en.wikipedia.org/wiki/Réka_Albert
https://openalex.org/A5012905268

---------------------------------------------------------------------- 
row 3/100
Christopher J. Schofield  (b. 1960, chemistry)   OpenAlex name: Christopher John Schofield
https://en.wikipedia.org/wiki/Christopher_J._Schofield
https://openalex.org/A5042398293

---------------------------------------------------------------------- 
row 4/100
William R. Simpson  (b. 1966, chemistry)   OpenAlex name: William Robert Simpson
https://en.wikipedia.org/wiki/William_R._Sim

## 3. Scoring the match check

Works with one labeller (accuracy only) or two (accuracy counts a match as correct only if **both** agree, plus agreement and Cohen's kappa).

A caveat for the paper: if almost every match is correct, **kappa is meaningless** (it collapses when one label dominates). Report raw agreement (e.g. "98/100") and list the disagreements instead.

In [5]:
def _load(kind, who):
    f = VAL / f"{kind}_labeller_{who}.csv"
    return pd.read_csv(f).set_index("qid") if f.exists() else None

def score_matches():
    A, B = _load("matches", "A"), _load("matches", "B")
    gender = ai.set_index("qid").gender
    out = {}
    for who, S in (("A", A), ("B", B)):
        if S is None or S.correct.notna().sum() == 0:
            print(f"Labeller {who}: nothing labelled yet"); continue
        d = S[S.correct.notna()].assign(gender=lambda x: x.index.map(gender))
        out[who] = d
        print(f"Labeller {who}: {len(d)} labelled, correct = {int(d.correct.sum())}, "
              f"profile complete = {d.profile_complete.mean():.0%}")
    if len(out) == 2:
        j = out["A"][["correct", "profile_complete", "gender"]].join(
            out["B"][["correct", "profile_complete"]], lsuffix="_A", rsuffix="_B", how="inner")
        print(f"\nBoth labelled: {len(j)} rows")
        for c in ("correct", "profile_complete"):
            a, b = j[c + "_A"].astype(int), j[c + "_B"].astype(int)
            kap = cohen_kappa_score(a, b) if a.nunique() + b.nunique() > 2 else float("nan")
            print(f"  {c}: raw agreement {(a == b).mean():.0%} ({(a == b).sum()}/{len(j)}), kappa = {kap:.2f}")
        j["both_correct"] = ((j.correct_A == 1) & (j.correct_B == 1)).astype(int)
        res, by = [], {}
        for g, d in j.groupby("gender"):
            k, n = int(d.both_correct.sum()), len(d); lo, hi = wilson(k, n)
            by[g] = (k, n); res.append({"gender": g, "n": n, "both_correct": k, "accuracy": k/n, "wilson_lo": lo, "wilson_hi": hi})
        display(pd.DataFrame(res).round(3))
        if len(by) == 2:
            t = [[k, n-k] for k, n in by.values()]
            print("Fisher exact p (women vs men accuracy):", round(fisher_exact(t)[1], 3),
                  "- with ~50 per group, differences under ~10 points cannot be detected")
        dis = j[(j.correct_A != j.correct_B)]
        if len(dis):
            print("\nDisagreements to discuss and settle together:")
            display(dis.join(ai.set_index("qid")[["title", "wikipedia_url", "openalex_url"]]))
    elif out:
        who = next(iter(out)); d = out[who]
        res = []
        for g, x in d.groupby("gender"):
            k, n = int(x.correct.sum()), len(x); lo, hi = wilson(k, n)
            res.append({"gender": g, "n": n, "correct": k, "accuracy": k/n, "wilson_lo": lo, "wilson_hi": hi})
        display(pd.DataFrame(res).round(3))
        print("Add the second labeller for agreement and kappa.")

score_matches()

Labeller A: 22 labelled, correct = 22, profile complete = 86%
Labeller B: nothing labelled yet


,gender,n,correct,accuracy,wilson_lo,wilson_hi
0,female,11,11,1.0,0.741,1.0
1,male,11,11,1.0,0.741,1.0


Add the second labeller for agreement and kappa.


## 4. The 300-sentence check (personal vs professional)

Same procedure: two people fill `sentences_labeller_A/B.csv` (`is_personal`, `is_professional`; a sentence can be both or neither). Context: the personal-life word list was **dropped** from the revised analysis because it found 51% of personal sentences on women's pages but only 33% on men's, so it cannot compare genders. The sentence labels still give you a validated number to report and show how you handled a failed measurement.

In [6]:
# label(WHO, "sentences")      # <- uncomment to label here (300 sentences, ~40 min); or edit the CSV directly

def score_sentences():
    A, B = _load("sentences", "A"), None
    f = VAL / "sentences_labeller_B.csv"
    if f.exists(): B = pd.read_csv(f)
    key = ["qid", "sent_idx"]
    ai_s = pd.read_csv(OLD / "results" / "sentences_to_label.csv").set_index(key)
    for who, S in (("A", pd.read_csv(VAL / "sentences_labeller_A.csv")), ("B", B)):
        if S is None or S.is_personal.notna().sum() == 0:
            print(f"Labeller {who}: nothing labelled yet"); continue
        S = S.set_index(key).dropna(subset=["is_personal", "is_professional"]).join(ai_s[["is_personal", "is_professional"]], rsuffix="_ai")
        print(f"Labeller {who}: {len(S)} sentences")
        for c in ("is_personal", "is_professional"):
            a, x = S[c].astype(int), S[c + "_ai"].astype(int)
            kap = cohen_kappa_score(a, x) if a.nunique() + x.nunique() > 2 else float("nan")
            print(f"  {c}: share positive {a.mean():.0%}; agreement with AI labels {(a == x).mean():.0%}, kappa {kap:.2f}")
    if B is not None and B.is_personal.notna().any():
        S = pd.read_csv(VAL / "sentences_labeller_A.csv").set_index(key).join(B.set_index(key)[["is_personal", "is_professional"]], rsuffix="_B").dropna()
        for c in ("is_personal", "is_professional"):
            a, b = S[c].astype(int), S[c + "_B"].astype(int)
            print(f"A vs B {c}: agreement {(a == b).mean():.0%}, kappa {cohen_kappa_score(a, b):.2f}")

score_sentences()

Labeller A: nothing labelled yet
Labeller B: nothing labelled yet


## 5. Oral-defence self-test

Anyone in the group can be picked, so everyone should be able to answer all of these. Numbers come from `CONCLUSIONS.md` (revised analysis); if you re-run the pipeline, re-check them there.

Run the next cell for the full list, or the one after for a random self-test (it asks, you answer out loud, press Enter to see the model answer).

In [7]:
QA = [('What is your research question, and why control for citations?', "Are women scientists' Wikipedia pages less embedded in the link network between scientists' pages, and written differently, than men's at equal citations, field and birth decade? Without the control, a gap could just reflect women having fewer citations; we measure the Wikipedia gap on top of any citation gap."), ("Why 'less embedded' and not 'less visible'?", "We measure links between scientists' pages, not readers. Pageviews were dropped (Wikipedia blocked our requests), so we cannot claim anything about readership."), ('Why physics and chemistry, born 1920–1980?', 'OpenAlex covers this period well, and the fields give enough women for comparisons. Biology was added in a single model (field × gender) on the full three-field network.'), ('How is the network built?', "Nodes are scientists' English Wikipedia pages; a directed edge A→B means A's page links to B's. Comparison network: 9,680 pages and 10,563 links. We also test a second definition: links in running prose only (no infobox, navbox, references, tables)."), ('What is the headline result?', "At equal citations, field and birth decade, women's pages receive about 40% fewer links from other scientists' pages: negative-binomial rate ratio 0.60, 95% CI [0.51, 0.70], n = 2,926 (606 women)."), ("Why did the headline change from '14% lower PageRank'?", 'Almost half the pages sit at the PageRank floor, so a log-PageRank ratio compresses a 40% gap in links into about 15%. Raw links received is closer to what is going on; PageRank (0.85 [0.80, 0.91]) stays as a supplement.'), ('How does the null model work?', 'We shuffle gender labels at random within strata of field × birth decade × citation quintile (network fixed), 1,000 times. The real gap is compared with the shuffled gaps: links received obs/expected 0.66, z = −4.6.'), ('What do the hurdle results add?', "Two parts: women are 9.2 percentage points less likely to receive any link [−13.8, −4.6], and given at least one link they receive about half as many (0.52 [0.40, 0.69]). Raw: 67% of women's pages get no incoming link versus 49% of men's."), ('What does the gap run through? Is it causal?', "Controls added one at a time: page length barely changes it (0.62); links on the page and the page's own outgoing links explain part (0.71 and 0.81). These are pathways, not confounders, since linking is often mutual. We cannot separate 'others link to them less' from 'their pages link out less', and we make no causal claim about why."), ('Is biology an exception?', "No, on the count scale: physics 0.59, chemistry 0.69, biology 0.64; joint test of different gaps p = 0.66. The earlier 'biology exception' came from the log-PageRank scale. What differs in biology is page length: women's pages about 19% longer."), ('Does the result survive robustness checks?', 'Yes: prose-only links 0.48, subfield strata 0.65, h-index 0.57, single OpenAlex profile 0.58, without Nobel laureates 0.61, without top 1% 0.60, born 1950+ 0.61, living people only 0.61.'), ('How do you know your impact data belong to the right people?', 'Matching used only ORCID or OpenAlex IDs from Wikidata, never names. A 100-match check (50 women, 50 men) found 100/100 correct (Wilson 95% ≈ 96–100%; per gender ≈ 93–100%), but those labels were AI-assisted, so we state that and have two blind human labellers filling their own sheets. Fill in the final human numbers here.'), ('Who is missing from your data (selection)?', 'Only people with an ORCID/OpenAlex ID can be compared at equal citations. Match rates rise sharply with birth decade (1920s: ~1–2%, 1970s: 62–72%), so matched people are younger and more often women (born 1970+: 33% of matched women vs 11% of matched men). Among the 6,747 unmatched pages the gap in links is the same size (rate ratio 0.50 vs 0.51), so matching does not create it. Not tested: whether it holds at equal citations among the unmatched (name-match script written, not run).'), ("RQ3: who links to women's pages?", "154 of 696 links to women's pages (22.1%) come from women's pages, which are 9.6% of linking pages. Expected under nulls: 12–16%; z between +4.3 and +9. The result gets stronger under the community × decade null."), ('Why was the RQ3 null changed?', 'The original null moved gender labels onto pages regardless of whether they link at all, which overstated how many links from women to expect. Shuffling within link community × decade over 10 Louvain partitions is stricter.'), ('Are women concentrated in particular communities?', 'They are present in all 16 large communities but unevenly (5.3% to 17.5% of each). We do not claim they are spread evenly. Louvain partitions vary between runs, so we report patterns over 10 partitions, not exact counts.'), ('RQ2: what does the classifier show?', 'TF-IDF + logistic regression on masked text (names, pronouns, gender words hidden), 5-fold CV: AUC 0.859 vs shuffled-gender null 0.576; within field × birth decade 0.846 vs 0.497. Field, decade, citations, length and deceased alone give 0.71. So pages differ systematically.'), ('Does a high AUC mean women are described differently?', 'Not necessarily. Strongest women-side words include British spellings and house-style verbs (whilst, honours, earned, completed), which fits pages written recently by few editors. Authorship and page age are live alternative explanations; testing them needs revision-history data.'), ('Which words distinguish the groups?', 'Women: education, career, earned, completed, recognition, doctorate. Men: quantum, theory, theoretical, papers, known, field. Held-out check: words picked on one half point the same way in the other half 93% of the time, so direction replicates but individual words should not be over-read.'), ('How common are explicit gender mentions?', "15.5% of women's pages ('first woman to…', 'female physicist') vs 0.1% of men's; adjusted difference 16.6 points [13.5, 19.7]. It shows framing, not intent."), ('Why was the personal-life result dropped?', "The word list found 51% of personal sentences on women's pages but 33% on men's, so it cannot compare genders. The 4.9% vs 2.8% difference in the top citation quintile stays an untested lead, not a result."), ('Does this prove Wikipedia is biased against women?', 'No. Links reflect editors, topics, page age and authorship. The design shows a gap at equal citations and constrains explanations; it does not identify a cause.'), ('What would change your conclusion?', 'If the gap vanished at equal impact, the inequality would sit upstream in citations, not Wikipedia (our own M2 test). Open threats: selection at equal citations among unmatched people, and page age/authorship.'), ('What did AI do in this project, and how did you check it?', 'Be specific and honest: AI assisted code and the first-round labels (100 matches, 300 sentences). Because that is circular, two humans relabelled blind and we report kappa and Wilson intervals. State exactly who did what; this must match the author-contributions section.')]

In [8]:
display(Markdown("\n\n".join(f"**{i}. {q}**\n\n{a}" for i, (q, a) in enumerate(QA, 1))))

**1. What is your research question, and why control for citations?**

Are women scientists' Wikipedia pages less embedded in the link network between scientists' pages, and written differently, than men's at equal citations, field and birth decade? Without the control, a gap could just reflect women having fewer citations; we measure the Wikipedia gap on top of any citation gap.

**2. Why 'less embedded' and not 'less visible'?**

We measure links between scientists' pages, not readers. Pageviews were dropped (Wikipedia blocked our requests), so we cannot claim anything about readership.

**3. Why physics and chemistry, born 1920–1980?**

OpenAlex covers this period well, and the fields give enough women for comparisons. Biology was added in a single model (field × gender) on the full three-field network.

**4. How is the network built?**

Nodes are scientists' English Wikipedia pages; a directed edge A→B means A's page links to B's. Comparison network: 9,680 pages and 10,563 links. We also test a second definition: links in running prose only (no infobox, navbox, references, tables).

**5. What is the headline result?**

At equal citations, field and birth decade, women's pages receive about 40% fewer links from other scientists' pages: negative-binomial rate ratio 0.60, 95% CI [0.51, 0.70], n = 2,926 (606 women).

**6. Why did the headline change from '14% lower PageRank'?**

Almost half the pages sit at the PageRank floor, so a log-PageRank ratio compresses a 40% gap in links into about 15%. Raw links received is closer to what is going on; PageRank (0.85 [0.80, 0.91]) stays as a supplement.

**7. How does the null model work?**

We shuffle gender labels at random within strata of field × birth decade × citation quintile (network fixed), 1,000 times. The real gap is compared with the shuffled gaps: links received obs/expected 0.66, z = −4.6.

**8. What do the hurdle results add?**

Two parts: women are 9.2 percentage points less likely to receive any link [−13.8, −4.6], and given at least one link they receive about half as many (0.52 [0.40, 0.69]). Raw: 67% of women's pages get no incoming link versus 49% of men's.

**9. What does the gap run through? Is it causal?**

Controls added one at a time: page length barely changes it (0.62); links on the page and the page's own outgoing links explain part (0.71 and 0.81). These are pathways, not confounders, since linking is often mutual. We cannot separate 'others link to them less' from 'their pages link out less', and we make no causal claim about why.

**10. Is biology an exception?**

No, on the count scale: physics 0.59, chemistry 0.69, biology 0.64; joint test of different gaps p = 0.66. The earlier 'biology exception' came from the log-PageRank scale. What differs in biology is page length: women's pages about 19% longer.

**11. Does the result survive robustness checks?**

Yes: prose-only links 0.48, subfield strata 0.65, h-index 0.57, single OpenAlex profile 0.58, without Nobel laureates 0.61, without top 1% 0.60, born 1950+ 0.61, living people only 0.61.

**12. How do you know your impact data belong to the right people?**

Matching used only ORCID or OpenAlex IDs from Wikidata, never names. A 100-match check (50 women, 50 men) found 100/100 correct (Wilson 95% ≈ 96–100%; per gender ≈ 93–100%), but those labels were AI-assisted, so we state that and have two blind human labellers filling their own sheets. Fill in the final human numbers here.

**13. Who is missing from your data (selection)?**

Only people with an ORCID/OpenAlex ID can be compared at equal citations. Match rates rise sharply with birth decade (1920s: ~1–2%, 1970s: 62–72%), so matched people are younger and more often women (born 1970+: 33% of matched women vs 11% of matched men). Among the 6,747 unmatched pages the gap in links is the same size (rate ratio 0.50 vs 0.51), so matching does not create it. Not tested: whether it holds at equal citations among the unmatched (name-match script written, not run).

**14. RQ3: who links to women's pages?**

154 of 696 links to women's pages (22.1%) come from women's pages, which are 9.6% of linking pages. Expected under nulls: 12–16%; z between +4.3 and +9. The result gets stronger under the community × decade null.

**15. Why was the RQ3 null changed?**

The original null moved gender labels onto pages regardless of whether they link at all, which overstated how many links from women to expect. Shuffling within link community × decade over 10 Louvain partitions is stricter.

**16. Are women concentrated in particular communities?**

They are present in all 16 large communities but unevenly (5.3% to 17.5% of each). We do not claim they are spread evenly. Louvain partitions vary between runs, so we report patterns over 10 partitions, not exact counts.

**17. RQ2: what does the classifier show?**

TF-IDF + logistic regression on masked text (names, pronouns, gender words hidden), 5-fold CV: AUC 0.859 vs shuffled-gender null 0.576; within field × birth decade 0.846 vs 0.497. Field, decade, citations, length and deceased alone give 0.71. So pages differ systematically.

**18. Does a high AUC mean women are described differently?**

Not necessarily. Strongest women-side words include British spellings and house-style verbs (whilst, honours, earned, completed), which fits pages written recently by few editors. Authorship and page age are live alternative explanations; testing them needs revision-history data.

**19. Which words distinguish the groups?**

Women: education, career, earned, completed, recognition, doctorate. Men: quantum, theory, theoretical, papers, known, field. Held-out check: words picked on one half point the same way in the other half 93% of the time, so direction replicates but individual words should not be over-read.

**20. How common are explicit gender mentions?**

15.5% of women's pages ('first woman to…', 'female physicist') vs 0.1% of men's; adjusted difference 16.6 points [13.5, 19.7]. It shows framing, not intent.

**21. Why was the personal-life result dropped?**

The word list found 51% of personal sentences on women's pages but 33% on men's, so it cannot compare genders. The 4.9% vs 2.8% difference in the top citation quintile stays an untested lead, not a result.

**22. Does this prove Wikipedia is biased against women?**

No. Links reflect editors, topics, page age and authorship. The design shows a gap at equal citations and constrains explanations; it does not identify a cause.

**23. What would change your conclusion?**

If the gap vanished at equal impact, the inequality would sit upstream in citations, not Wikipedia (our own M2 test). Open threats: selection at equal citations among unmatched people, and page age/authorship.

**24. What did AI do in this project, and how did you check it?**

Be specific and honest: AI assisted code and the first-round labels (100 matches, 300 sentences). Because that is circular, two humans relabelled blind and we report kappa and Wilson intervals. State exactly who did what; this must match the author-contributions section.

In [ ]:
def quiz(n=5, seed=None):
    rnd = random.Random(seed)
    for q, a in rnd.sample(QA, min(n, len(QA))):
        print("\nQ:", q)
        try:
            input("   (answer out loud, then press Enter) ")
        except Exception:
            print("   (run this cell interactively in VS Code)"); return
        print("A:", a)

quiz(5)


Q: How do you know your impact data belong to the right people?
A: Matching used only ORCID or OpenAlex IDs from Wikidata, never names. A 100-match check (50 women, 50 men) found 100/100 correct (Wilson 95% ≈ 96–100%; per gender ≈ 93–100%), but those labels were AI-assisted, so we state that and have two blind human labellers filling their own sheets. Fill in the final human numbers here.

Q: What did AI do in this project, and how did you check it?
A: Be specific and honest: AI assisted code and the first-round labels (100 matches, 300 sentences). Because that is circular, two humans relabelled blind and we report kappa and Wilson intervals. State exactly who did what; this must match the author-contributions section.

Q: How common are explicit gender mentions?
A: 15.5% of women's pages ('first woman to…', 'female physicist') vs 0.1% of men's; adjusted difference 16.6 points [13.5, 19.7]. It shows framing, not intent.

Q: Are women concentrated in particular communities?
